<a href="https://colab.research.google.com/github/lsgrep/serv/blob/main/notebooks/11_attention_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 11 — Attention, from the arithmetic up

**The claim you should be able to make when you finish:** *"I can derive the KV
cache instead of quoting it — including why a cache is sound at all, why prefill
and decode hit opposite bottlenecks, and what flash attention does and does not
buy."*

Every other lab here consumes facts about attention. The KV formula. GQA
shrinking it. Prefill compute bound, decode bandwidth bound. Long context being
expensive. Those are usually memorised, and memorised facts fail on the second
follow-up — *"why?"* — which is exactly the question a good interviewer asks.

This lab derives them. Tiny tensors, printed matrices, numpy. Forty minutes.

### Do this one before lab 3

The ladder is numbered in the order the labs were built, not the order to learn
them. This is the foundation under [lab 3](03_kv_math_and_toy_engine.ipynb),
which does the napkin math, and under everything downstream of that.

### The five things it builds

1. Attention is a weighted lookup — three lines of arithmetic.
2. **Causal masking is what makes a cache possible at all.** Not an optimisation:
   a property.
3. Prefill and decode are the same function at different shapes.
4. The score matrix is the quadratic term, and it is *memory* before it is compute.
5. Positions are baked into keys — which is why prefix caching is a *prefix* cache.

In [ ]:
# Cell 1 — bootstrap. No GPU, no model download: this is numpy on tiny tensors.
REPO, BRANCH = "https://github.com/lsgrep/serv.git", "main"

import os, subprocess, sys

if not os.path.isdir("serv"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "serv", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("serv"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "numpy"], check=True)

import numpy as np
from servlab import attention as at, napkin as nk
from servlab.plots import use_style, SERIES, STATUS
use_style()
np.set_printoptions(precision=3, suppress=True, linewidth=110)
print("ready")

## 1. Attention is a weighted lookup

Three steps. Score every query against every key, turn the scores into weights
that sum to one, take that weighted average of the values.

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V$$

The $\sqrt{d}$ exists so dot products do not grow with dimension and shove the
softmax into a one-hot corner. Take it out below and watch the weights collapse
onto a single token — that is what "unscaled" attention actually does.

In [ ]:
rng = np.random.default_rng(0)
n, d = 5, 8
Q, K, V = (rng.normal(size=(n, d)) for _ in range(3))

scores = Q @ K.T / np.sqrt(d)
weights = at.softmax(scores)
out = weights @ V

print("scores (query x key):\n", scores, "\n")
print("weights — every row sums to 1:\n", weights, "\n")
print("row sums:", weights.sum(axis=1))
print("output shape:", out.shape, "— one vector per query, same width as V")

In [ ]:
# Without the 1/sqrt(d) scale, at a realistic head dim.
big_d = 128
Qb, Kb = (rng.normal(size=(3, big_d)) for _ in range(2))
unscaled = at.softmax(Qb @ Kb.T)
scaled = at.softmax(Qb @ Kb.T / np.sqrt(big_d))
print("unscaled max weight per row:", unscaled.max(axis=1).round(3))
print("scaled   max weight per row:", scaled.max(axis=1).round(3))
print("\nUnscaled, the softmax has already decided. There is nothing left to")
print("learn from the other tokens, and gradients through it are near zero.")

## 2. Causal masking — and why it is what makes a cache possible

Autoregressive generation needs token *i* to see only tokens ≤ *i*. Set the rest
to −∞ before the softmax and they get exactly zero weight.

Now the consequence almost nobody states, and it is the whole justification for
the KV cache:

> Because attention is causal, **the key and value computed for token *i* are
> never revised.** Token 50 arriving does not change token 3's K or V. They are
> immutable the moment they are computed.

That is why an append-only cache is *correct*, not merely convenient. If
attention were bidirectional — as in an encoder — every new token would change
every earlier representation, and no such cache could exist.

In [ ]:
mask = at.causal_mask(n)
print("mask (True = allowed):\n", mask.astype(int), "\n")

masked_out, masked_w = at.attention(Q, K, V, mask=mask, return_weights=True)
print("weights under the mask:\n", masked_w.round(3))
print("\nRow 0 attends only to itself. Row 4 sees everything. Upper triangle is")
print("exactly zero — not small, zero.")

In [ ]:
# The immutability claim, tested rather than asserted: extend the sequence and
# check that earlier rows' outputs do not move.
Q7, K7, V7 = (np.concatenate([m, rng.normal(size=(2, d))]) for m in (Q, K, V))
short = at.attention(Q, K, V, mask=at.causal_mask(5))
long = at.attention(Q7, K7, V7, mask=at.causal_mask(7))

print("first 5 outputs identical after appending 2 tokens:",
      np.allclose(short, long[:5]))
print("\nThat invariance is the licence to cache. Everything else in serving —")
print("paging, prefix reuse, continuous batching — is built on top of it.")

In [ ]:
# What the weights look like. A heatmap, one hue light-to-dark: this is a
# magnitude, so a single sequential ramp, never a rainbow.
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

BLUE_RAMP = LinearSegmentedColormap.from_list(
    "seq_blue", ["#f4f8fe", "#cde2fb", "#9ec5f4", "#5598e7", "#2a78d6", "#184f95", "#0d366b"])

longer = 12
Ql, Kl, Vl = (rng.normal(size=(longer, d)) for _ in range(3))
_, wl = at.attention(Ql, Kl, Vl, mask=at.causal_mask(longer), return_weights=True)

fig, ax = plt.subplots(figsize=(5.6, 5))
im = ax.imshow(wl, cmap=BLUE_RAMP, vmin=0, vmax=wl.max())
ax.set_xlabel("key position (what is attended to)")
ax.set_ylabel("query position (which token is asking)")
ax.set_title("causal attention weights")
fig.colorbar(im, ax=ax, fraction=0.046, label="attention weight")
plt.show()

print("The triangle is the mask. With random weights the mass spreads evenly;")
print("in a trained model these rows are spiky, and a surprising amount lands")
print(f"on position 0 — the attention-sink effect. Here: {at.sink_share(wl):.1%}.")

## 3. Prefill and decode: same function, opposite bottlenecks

Now the shape argument that the rest of the ladder rests on.

| | Q shape | K, V shape | Reads | Arithmetic |
|---|---|---|---|---|
| **Prefill** | `[n, d]` | `[n, d]` | weights once | `O(n²d)` — a big matmul |
| **Decode** | `[1, d]` | `[n, d]` | weights once, KV every step | `O(nd)` — a matrix-vector product |

Same code. In prefill there are *n* query rows to amortise the weight read
across, so the GPU's ALUs are busy. In decode there is **one**, so the same
weight read produces a single token and the card spends its time waiting on
memory.

That asymmetry is the origin of continuous batching (find more query rows),
speculative decoding (get more tokens per weight read), and the entire
prefill/decode split in modern serving stacks.

In [ ]:
w = at.AttentionWeights(d_model=64, n_heads=8, n_kv_heads=2, seed=0)
x = rng.normal(size=(12, 64))
print(f"d_model {w.d_model}, {w.n_heads} query heads over {w.n_kv_heads} KV heads "
      f"(GQA {w.gqa_ratio:g}:1), head dim {w.d_head}")

q, k, v = w.project(x)
print(f"\nQ {q.shape}   K {k.shape}   V {v.shape}")
print("Note K and V are narrower — that is GQA, and it happens in the")
print("projection, not in the attention maths.")

In [ ]:
# Prefill the first 8 tokens, then decode 4 one at a time.
out, cache = at.prefill(x[:8], w)
print(f"after prefill:  cache holds {cache.length} tokens, {cache.nbytes:,.0f} bytes")

for t in range(8, 12):
    step, cache = at.decode_step(x[t], w, cache)
    print(f"  decode step {t}: cache {cache.length} tokens, {cache.nbytes:,.0f} bytes")

full = at.mha_forward(x, w)
print(f"\ncached decoding == full recomputation: {np.allclose(full[-1], step[0])}")
print("\nThat equality is the correctness proof of the KV cache. If it were ever")
print("false, caching would be a bug rather than an optimisation.")

In [ ]:
# The formula everyone quotes, now derived from a thing we built.
print(f"cache bytes per token, one layer: 2 x {w.n_kv_heads} kv heads x "
      f"{w.d_head} head dim x 2 bytes = {cache.bytes_per_token():.0f}")

spec = nk.MODELS["llama-3.3-70b"]
per_layer = at.KVCache(spec.n_kv_heads, spec.head_dim, dtype_bytes=2).bytes_per_token()
print(f"\nLlama-3.3-70B, one layer: {per_layer:,.0f} bytes/token")
print(f"x {spec.n_layers} layers        = {per_layer * spec.n_layers:,.0f} bytes/token")
print(f"napkin.kv_bytes_per_token   = {nk.kv_bytes_per_token(spec):,.0f} bytes/token")
print(f"\nagree: {per_layer * spec.n_layers == nk.kv_bytes_per_token(spec)}")

## 4. MHA → MQA → GQA is one tensor shape

Multi-head attention gives every query head its own K and V. Multi-query gives
them all **one** shared pair. Grouped-query is the compromise: heads share in
groups.

At inference the entire mechanism is a `repeat`: the cache stores `n_kv_heads`
of them, and the kernel broadcasts each across its group. Nothing is
approximated, nothing is recomputed — the cache is smaller by exactly
`n_heads / n_kv_heads`.

That is why GQA is a *serving* innovation. It buys nothing in FLOPs. It buys
concurrency.

In [ ]:
kv = np.arange(2 * 3 * 4).reshape(2, 3, 4)      # 2 KV heads
rep = at.repeat_kv(kv, 8)                        # 8 query heads
print(f"{kv.shape} -> {rep.shape}")
print("query heads 0-3 all use KV head 0:", np.array_equal(rep[0], rep[3]), np.array_equal(rep[3], kv[0]))
print("query heads 4-7 all use KV head 1:", np.array_equal(rep[4], kv[1]))

In [ ]:
# Read the ratios off a real config rather than a toy one, so the numbers are
# the model's own. Llama-3.3-70B ships 64 query heads over 8 KV heads.
spec = nk.MODELS["llama-3.3-70b"]
mha = nk.ModelSpec(**{**spec.__dict__, "n_kv_heads": spec.n_heads})

print(f"{'variant':<22}{'kv heads':>10}{'bytes/token':>14}{'vs MHA':>9}{'32K context':>14}")
for label, n_kv in (("MHA", spec.n_heads),
                    ("GQA 8:1 (as shipped)", spec.n_kv_heads),
                    ("MQA", 1)):
    v = nk.ModelSpec(**{**spec.__dict__, "n_kv_heads": n_kv})
    per_tok = nk.kv_bytes_per_token(v)
    print(f"{label:<22}{n_kv:>10}{nk.human_bytes(per_tok):>14}"
          f"{nk.kv_bytes_per_token(mha) / per_tok:>8.0f}x"
          f"{nk.human_bytes(nk.kv_bytes(v, 32768)):>14}")

print("\nA 32K conversation costs 10 GiB of KV on this model as shipped. Built with")
print("plain multi-head attention the same model would need 80 GiB — an entire")
print("H100 for one conversation, before the weights. GQA is why long-context")
print("serving exists, and it cost nothing in FLOPs to get.")

## 5. The quadratic term is memory before it is compute

The score matrix is `n × n` per head. Its **FLOPs** grow quadratically, which
everyone knows. What bites first is that its **bytes** grow quadratically too,
and a materialised score matrix at long context is larger than the model.

In [ ]:
print(f"{'context':>9}{'score matrix (32 heads, fp16)':>32}{'attn share of layer FLOPs':>28}")
for ctx in (512, 2048, 8192, 32768, 131072):
    b = at.score_matrix_bytes(ctx, n_heads=32)
    share = at.attention_share(ctx, d_model=4096, n_heads=32, n_kv_heads=8)
    print(f"{ctx:>9,}{nk.human_bytes(b):>32}{share:>27.1%}")

cross = at.quadratic_crossover(4096, 32, 8)
print(f"\nAttention passes half of a layer's FLOPs at ~{cross:,} tokens of context.")
print("Below that, context is nearly free and the model is 'just matmuls'.")
print("Above it, doubling context more than doubles the cost.")

In [ ]:
import matplotlib.pyplot as plt

ctxs = [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(ctxs, [at.score_matrix_bytes(c, 32) / 1024**3 for c in ctxs],
             marker="o", color=SERIES[0], label="materialised scores")
axes[0].axhline(80, color=STATUS["critical"], linestyle=":", linewidth=1.5)
axes[0].annotate("an 80 GB card", xy=(ctxs[0], 80), xytext=(0, 5),
                 textcoords="offset points", color=STATUS["critical"], fontsize=9)
axes[0].set_xscale("log", base=2); axes[0].set_yscale("log")
axes[0].set_xlabel("context length"); axes[0].set_ylabel("GiB")
axes[0].set_title("the score matrix, if you materialise it")
axes[0].legend(loc="upper left")

axes[1].plot(ctxs, [at.attention_share(c, 4096, 32, 8) * 100 for c in ctxs],
             marker="o", color=SERIES[0])
axes[1].axvline(cross, color=STATUS["warning"], linestyle="--", linewidth=1.5)
axes[1].annotate(f"half at ~{cross//1024}K", xy=(cross, 20), xytext=(6, 0),
                 textcoords="offset points", color=STATUS["warning"], fontsize=9)
axes[1].set_xscale("log", base=2)
axes[1].set_xlabel("context length"); axes[1].set_ylabel("% of layer FLOPs in attention")
axes[1].set_title("when attention stops being a rounding error")
plt.tight_layout(); plt.show()

## 6. Flash attention: what it actually does

The blocker to *not* materialising that matrix is softmax — it needs the whole
row to normalise. Or so it appears.

**Online softmax** removes the blocker. Walk the row in tiles carrying a running
max `m` and running sum `denom`; when a tile's max exceeds `m`, rescale what you
have accumulated by `exp(old_m − new_m)` and continue. The answer is identical
and you never held more than a tile.

That is the whole idea. Flash attention tiles Q, K and V, keeps the tile in
on-chip SRAM, and never writes the `n × n` matrix to HBM.

**It saves no FLOPs.** The same multiplications happen. What changes is memory
traffic — which, since attention was memory bound, is what was costing you.

In [ ]:
row = rng.normal(size=(2, 17)) * 5
print("online softmax == one-pass softmax:",
      np.allclose(at.online_softmax(row, block_size=4), at.softmax(row)))
print("...at every block size:",
      all(np.allclose(at.online_softmax(row, block_size=b), at.softmax(row))
          for b in (1, 2, 3, 8, 32)))

In [ ]:
# The full tiled attention, against the naive version.
Qf, Kf, Vf = (rng.normal(size=(16, 8)) for _ in range(3))
m16 = at.causal_mask(16)
print("flash == naive (causal):",
      np.allclose(at.flash_attention(Qf, Kf, Vf, mask=m16, block_size=4),
                  at.attention(Qf, Kf, Vf, mask=m16)))

In [ ]:
naive_b = at.score_matrix_bytes(8192, n_heads=32)
tiled_b = at.flash_workspace_bytes(128, n_heads=32)
print(f"8K context, 32 heads, fp16:")
print(f"  materialised scores  {nk.human_bytes(naive_b)}")
print(f"  one 128x128 tile     {nk.human_bytes(tiled_b)}")
print(f"  ratio                {naive_b / tiled_b:,.0f}x less resident")
print(f"\n  FLOPs, either way:   {at.attention_flops(8192, 4096):.3e}  — unchanged")
print("\nThe sentence to be able to say: flash attention is a memory-traffic")
print("optimisation, not an arithmetic one. It is why long context became")
print("practical, and it is not why attention got faster to compute.")

## 7. Positions are baked into the keys

RoPE rotates position *into* the query and key vectors, so a score depends on
the **distance** between two positions rather than their absolute values.

The serving consequence is one people get wrong about prefix caching:

* A shared **system prompt** sits at positions 0..n on every request. Its keys
  are identical every time, so they are reusable verbatim. This is why prefix
  caching works.
* A shared **paragraph in the middle** of two different documents sits at
  different offsets. Its cached keys carry the wrong rotation and are worthless
  to the second document.

Prefix caching is a prefix cache because of this, not because of an
implementation shortcut.

In [ ]:
r = at.relative_score_shift(d_head=8, gap=3)
print("same query/key pair, same gap of 3, at three different absolute positions:")
for s in r["same_gap_scores"]:
    print(f"    {s:.6f}")
print(f"  spread: {r['spread']:.2e}  — identical, because only the distance matters\n")
print(f"the same pair with the key reused at the wrong offset: {r['wrong_offset_score']:.6f}")
print("\nDifferent number, so different attention weights, so a different answer.")
print("A cached key is only valid at the position it was computed for.")

## 8. Now reread the rest of the ladder

Every claim the other labs make should now be derivable rather than recalled:

| Elsewhere | Because, from this lab |
|---|---|
| `2 × layers × kv_heads × head_dim × bytes` | K and V, per layer, at the *KV* head count — §3 and §4 |
| The cache is valid at all | Causal masking makes earlier K/V immutable — §2 |
| Decode is bandwidth bound | One query row per full weight read — §3 |
| Prefill is compute bound | `n` query rows amortise the same read — §3 |
| GQA shrinks the cache 4× | The KV projections are narrower; the kernel repeats — §4 |
| FP8 KV doubles concurrency | The formula is linear in bytes per element — §3 |
| Long context is expensive | Attention passes half of layer FLOPs at ~21K — §5 |
| Flash attention helps | It removes the materialised `n × n`, not the FLOPs — §6 |
| Prefix caching only works on prefixes | Positions are rotated into the keys — §7 |

## What to be able to say afterwards

1. **Why a KV cache is correct**, not just useful — and that an encoder could
   not have one.
2. **Prefill and decode are one function at two shapes**, and the shape is what
   decides the bottleneck.
3. **GQA is a repeat**, applied at inference, that buys concurrency and not speed.
4. **Flash attention is memory traffic, not arithmetic.** Volunteering that
   distinction is a strong signal, because most people state it the other way.
5. **Where the quadratic term starts to matter** for a model you can name.
6. **Why prefix caching is a prefix cache**, from RoPE rather than from folklore.

**Next:** [lab 3](03_kv_math_and_toy_engine.ipynb) turns these into capacity
planning, then builds a scheduler on top.